# 🏏 Feature Engineering for IPL Performance Prediction

In this notebook we will:

- Load the cleaned dataset
- Create target variable: `high_score` (1 if runs ≥ 50 else 0)
- Remove data leakage columns
- Select useful features
- Prepare final dataset for Machine Learning
- Save `final_ml.csv`


## 🧠 CODE CELL 1 — Imports & Load Data

In [3]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv("../data/cleaned_batting_card.csv")

df.head()


,season,match_id,match_name,home_team,away_team,venue,city,country,current_innings,innings_id,...,minutes,fours,sixes,strikerate,captain,isnotout,runningscore,runningover,shorttext,commentary
0,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,0,0.0,0.0,16.66,False,False,0,2.2,b Mohammed Shami,<strong>Shami breaches the defences of Conway<...
1,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,0,4.0,9.0,184.00,False,False,0,17.1,c Shubman Gill b Joseph,"Joseph misses the yorker, but is still a hard-..."
2,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,0,4.0,1.0,135.29,False,False,0,5.5,c &dagger;Saha b Rashid Khan,"<strong>Rashid strikes back, Moeen has nicked ..."
3,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,0,1.0,0.0,116.66,False,False,0,7.4,c &dagger;Saha b Rashid Khan,<strong>Rashid takes out both Moeen and Stokes...
4,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,0,0.0,1.0,100.00,False,False,0,12.5,b Little,Maiden IPL wicket for Josh Little. Rayudu dash...


## 🔍 Dataset Overview


In [4]:
df.shape, df.columns


((15720, 24),
 Index(['season', 'match_id', 'match_name', 'home_team', 'away_team', 'venue',
        'city', 'country', 'current_innings', 'innings_id', 'name', 'fullname',
        'runs', 'ballsfaced', 'minutes', 'fours', 'sixes', 'strikerate',
        'captain', 'isnotout', 'runningscore', 'runningover', 'shorttext',
        'commentary'],
       dtype='object'))

## 🎯 Creating Target Variable

We define:

- `high_score = 1` → Batsman scored 50 or more runs
- `high_score = 0` → Otherwise


In [5]:
df["high_score"] = (df["runs"] >= 50).astype(int)

df["high_score"].value_counts()


high_score
0    14070
1     1650
Name: count, dtype: int64

## 🚨 Removing Data Leakage Columns

We must remove columns that directly or indirectly reveal the target:

- `runs`
- `fours`
- `sixes`
- `strikerate`

Because they are only known AFTER the innings is played.


In [6]:
leakage_cols = ["runs", "fours", "sixes", "strikerate"]

df = df.drop(columns=leakage_cols)

df.head()


,season,match_id,match_name,home_team,away_team,venue,city,country,current_innings,innings_id,...,fullname,ballsfaced,minutes,captain,isnotout,runningscore,runningover,shorttext,commentary,high_score
0,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Devon Conway,6.0,0,False,False,0,2.2,b Mohammed Shami,<strong>Shami breaches the defences of Conway<...,0
1,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Ruturaj Gaikwad,50.0,0,False,False,0,17.1,c Shubman Gill b Joseph,"Joseph misses the yorker, but is still a hard-...",1
2,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Moeen Ali,17.0,0,False,False,0,5.5,c &dagger;Saha b Rashid Khan,"<strong>Rashid strikes back, Moeen has nicked ...",0
3,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Ben Stokes,6.0,0,False,False,0,7.4,c &dagger;Saha b Rashid Khan,<strong>Rashid takes out both Moeen and Stokes...,0
4,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Ambati Rayudu,12.0,0,False,False,0,12.5,b Little,Maiden IPL wicket for Josh Little. Rayudu dash...,0


## 🧹 Dropping Unnecessary Columns

We remove identifiers like:

- Player name
- Match id
- Any purely ID columns


In [7]:
id_cols = ["player"] if "player" in df.columns else []

df = df.drop(columns=id_cols, errors="ignore")

df.head()


,season,match_id,match_name,home_team,away_team,venue,city,country,current_innings,innings_id,...,fullname,ballsfaced,minutes,captain,isnotout,runningscore,runningover,shorttext,commentary,high_score
0,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Devon Conway,6.0,0,False,False,0,2.2,b Mohammed Shami,<strong>Shami breaches the defences of Conway<...,0
1,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Ruturaj Gaikwad,50.0,0,False,False,0,17.1,c Shubman Gill b Joseph,"Joseph misses the yorker, but is still a hard-...",1
2,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Moeen Ali,17.0,0,False,False,0,5.5,c &dagger;Saha b Rashid Khan,"<strong>Rashid strikes back, Moeen has nicked ...",0
3,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Ben Stokes,6.0,0,False,False,0,7.4,c &dagger;Saha b Rashid Khan,<strong>Rashid takes out both Moeen and Stokes...,0
4,2023,1359475,GT v CSK,GT,CSK,"Narendra Modi Stadium, Motera, Ahmedabad",Ahmedabad,India,CSK,1,...,Ambati Rayudu,12.0,0,False,False,0,12.5,b Little,Maiden IPL wicket for Josh Little. Rayudu dash...,0


## ✅ Final Dataset Check


In [8]:
df.shape, df.isnull().sum()


((15720, 21),
 season             0
 match_id           0
 match_name         0
 home_team          0
 away_team          0
 venue              0
 city               0
 country            0
 current_innings    0
 innings_id         0
 name               0
 fullname           0
 ballsfaced         0
 minutes            0
 captain            0
 isnotout           0
 runningscore       0
 runningover        0
 shorttext          0
 commentary         0
 high_score         0
 dtype: int64)

💾 Saving Final ML Data

In [9]:
df.to_csv("../data/final_ml.csv", index=False)

print("Saved to data/final_ml.csv")


Saved to data/final_ml.csv
